<a href="https://colab.research.google.com/github/nanaafuacisse-cmyk/project-1-/blob/main/Bonolo_3_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import csv
import math
import random

random.seed(42)

In [ ]:
# Utilities
utilities = [
    (1, "Electricity Company of Ghana", "ECG", "ECG", "Distribution", "Ghana", "Y"),
    (2, "Northern Electricity Distribution Company", "NEDCo", "NED", "Distribution", "Ghana", "Y"),
    (3, "Ghana Grid Company", "GRIDCo", "GRD", "Transmission", "Ghana", "Y"),
    (4, "Volta River Authority", "VRA", "VRA", "Generation", "Ghana", "Y"),
    (5, "Compagnie Ivoirienne d'Electricite", "CIE", "CIE", "Distribution", "Cote d'Ivoire", "Y"),
    (6, "Communaute Electrique du Benin", "CEB", "CEB", "Transmission", "Togo/Benin", "Y"),
    (7, "Societe Beninoise d'Energie Electrique", "SBEE", "SBE", "Distribution", "Benin", "Y"),
    (8, "Electricite de Guinee", "EDG", "EDG", "Distribution", "Guinea", "N"),
    (9, "Sonabel", "SONABEL", "SNB", "Distribution", "Burkina Faso", "Y"),
    (10, "Enclave Power Company", "EPC", "EPC", "Generation", "Ghana", "N"),
]

In [ ]:
ghana_regions = {
    "Greater Accra": [("Achimota", 5.614, -0.224), ("Tema", 5.669, -0.017),
                       ("Mallam", 5.560, -0.298), ("Legon", 5.650, -0.186),
                       ("Kaneshie", 5.560, -0.238), ("Aboadze Junction", 5.590, -0.150)],
    "Ashanti": [("Kumasi Central", 6.688, -1.624), ("Ejisu", 6.719, -1.463),
                ("Obuasi", 6.202, -1.663), ("Mampong", 7.062, -1.400), ("Konongo", 6.618, -1.219)],
    "Western": [("Takoradi", 4.895, -1.759), ("Aboadze", 4.985, -1.766),
                ("Tarkwa", 5.301, -1.994), ("Axim", 4.867, -2.241)],
    "Central": [("Cape Coast", 5.106, -1.246), ("Winneba", 5.352, -0.622),
                ("Kasoa", 5.533, -0.416), ("Assin Fosu", 5.699, -1.492)],
    "Eastern": [("Koforidua", 6.094, -0.259), ("Akosombo", 6.300, 0.055),
                ("Nkawkaw", 6.550, -0.767), ("Suhum", 6.041, -0.451)],
    "Volta": [("Ho", 6.611, 0.471), ("Kpong", 6.150, 0.100),
              ("Hohoe", 7.152, 0.472), ("Sogakope", 6.007, 0.573)],
    "Bono": [("Sunyani", 7.339, -2.326), ("Techiman", 7.590, -1.938), ("Berekum", 7.453, -2.585)],
    "Northern": [("Tamale", 9.403, -0.842), ("Yendi", 9.442, -0.011), ("Savelugu", 9.625, -0.826)],
    "Upper East": [("Bolgatanga", 10.787, -0.851), ("Bawku", 11.058, -0.243)],
    "Upper West": [("Wa", 10.061, -2.501)],
}

In [ ]:
cross_border = [
    ("Bolgatanga Interconnection", "Burkina Faso border", 11.20, -0.75),
    ("Elubo Border Station", "Cote d'Ivoire border", 5.20, -2.85),
    ("Aflao Border Station", "Togo border", 6.12, 1.19),
    ("Lome Transmission Hub", "Togo", 6.13, 1.22),
    ("Cotonou Transmission Hub", "Benin", 6.37, 2.43),
    ("Abidjan Transmission Hub", "Cote d'Ivoire", 5.35, -4.00),
    ("Bobo-Dioulasso Hub", "Burkina Faso", 11.18, -4.30),
    ("Conakry Transmission Hub", "Guinea", 9.64, -13.58),
]


In [ ]:
voltage_levels = [11, 33, 69, 161, 330]
substations = []
sid = 1
name_to_id = {}

In [ ]:
for region, places in ghana_regions.items():
    for name, lat, lon in places:
        voltage = random.choice(voltage_levels)
        if voltage >= 161:
            sub_type = "Transmission"
        elif voltage == 69:
            sub_type = "Bulk Supply Point"
        else:
            sub_type = "Distribution"
        if sub_type != "Distribution":
            capacity = round(random.uniform(15, 400), 1)
        else:
            capacity = round(random.uniform(5, 60), 1)
        status = "Active" if random.random() > 0.05 else "Inactive"
        substations.append([
            sid, f"{name} Substation", name, region, "Ghana",
            round(lat + random.uniform(-0.01, 0.01), 4),
            round(lon + random.uniform(-0.01, 0.01), 4),
            voltage, capacity, random.randint(1965, 2023), sub_type, status,
        ])
        name_to_id[name] = sid
        sid += 1

for name, country, lat, lon in cross_border:
    voltage = random.choice([161, 330])
    capacity = round(random.uniform(100, 500), 1)
    substations.append([
        sid, name, name, country, country.split()[0],
        round(lat, 4), round(lon, 4), voltage, capacity,
        random.randint(1980, 2020), "Transmission", "Active",
    ])
    name_to_id[name] = sid
    sid += 1

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

sub_by_id = {row[0]: row for row in substations}
lines = []
lid = 1
seen_pairs = set()

In [ ]:
for region, places in ghana_regions.items():
    ids_in_region = [name_to_id[n] for n, _, _ in places]
    for i, a in enumerate(ids_in_region):
        for b in ids_in_region[i + 1:]:
            if random.random() < 0.55:
                pair = tuple(sorted((a, b)))
                if pair in seen_pairs:
                    continue
                seen_pairs.add(pair)
                utility_id = random.choice([1, 2, 3])
                sub_a, sub_b = sub_by_id[a], sub_by_id[b]
                dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * random.uniform(1.05, 1.3), 1)
                voltage = min(sub_a[7], sub_b[7])
                lines.append([
                    lid, utility_id, a, sub_a[1], b, sub_b[1],
                    voltage, dist, round(random.uniform(20, 300), 1),
                    "Active" if random.random() > 0.08 else "Under Maintenance",
                    random.choice(["Overhead", "Underground"]),
                ])
                lid += 1

In [ ]:
region_hub = {r: name_to_id[places[0][0]] for r, places in ghana_regions.items()}
region_names = list(region_hub.keys())
for i in range(len(region_names) - 1):
    a = region_hub[region_names[i]]
    b = region_hub[region_names[i + 1]]
    sub_a, sub_b = sub_by_id[a], sub_by_id[b]
    dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * 1.15, 1)
    lines.append([
        lid, 3, a, sub_a[1], b, sub_b[1], 330, dist,
        round(random.uniform(200, 600), 1), "Active", "Overhead",
    ])
    lid += 1

In [ ]:
border_links = [
    ("Bolgatanga", "Bolgatanga Interconnection", 9),
    ("Bolgatanga Interconnection", "Bobo-Dioulasso Hub", 9),
    ("Elubo Border Station", "Kumasi Central", 5),
    ("Elubo Border Station", "Abidjan Transmission Hub", 5),
    ("Aflao Border Station", "Tema", 6),
    ("Aflao Border Station", "Lome Transmission Hub", 6),
    ("Lome Transmission Hub", "Cotonou Transmission Hub", 6),
]
for src_name, dst_name, utility_id in border_links:
    if src_name not in name_to_id or dst_name not in name_to_id:
        continue
    a, b = name_to_id[src_name], name_to_id[dst_name]
    sub_a, sub_b = sub_by_id[a], sub_by_id[b]
    dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * 1.1, 1)
    lines.append([
        lid, utility_id, a, sub_a[1], b, sub_b[1], 330, dist,
        round(random.uniform(150, 400), 1), "Active", "Overhead",
    ])
    lid += 1

In [ ]:
with open("utilities.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Utility ID", "Name", "Alias", "Code", "Type", "Country", "Active"])
    w.writerows(utilities)

with open("substations.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Substation ID", "Name", "Short Name", "Region", "Country", "Latitude", "Longitude",
                "Voltage (kV)", "Capacity (MVA)", "Commissioning Year", "Type", "Status"])
    w.writerows(substations)

with open("lines.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Line ID", "Utility ID", "Source Substation ID", "Source Substation",
                "Destination Substation ID", "Destination Substation", "Voltage (kV)",
                "Length (km)", "Capacity (MVA)", "Status", "Line Type"])
    w.writerows(lines)

print("utilities:", len(utilities), "rows")
print("substations:", len(substations), "rows")
print("lines:", len(lines), "rows")

utilities: 10 rows
substations: 44 rows
lines: 55 rows


In [ ]:
!pip install streamlit plotly networkx -q
!npm install -g localtunnel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 84.6 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

**BUILDING DASHBOARD**

In [ ]:
!pip install dash plotly networkx pandas -q
!npm install -g localtunnel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 43.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦
changed 22 packages in 782ms
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [ ]:
%%writefile app.py

# Dashboard Development using Dash

import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, dash_table, Input, Output


#  Load the data

utilities = pd.read_csv("utilities.csv")
substations = pd.read_csv("substations.csv")
lines = pd.read_csv("lines.csv")
lines = lines.merge(utilities[["Utility ID", "Alias"]], on="Utility ID", how="left")

region_list = sorted(substations["Region"].unique())
voltage_list = sorted(substations["Voltage (kV)"].unique())
utility_list = sorted(utilities["Alias"].unique())

# A helper function that applies the filters

def filter_data(regions, voltages, utils):
    filtered_substations = substations[
        (substations["Region"].isin(regions)) &
        (substations["Voltage (kV)"].isin(voltages))
    ]
    sub_names = filtered_substations["Name"].tolist()
    filtered_lines = lines[
        (lines["Alias"].isin(utils)) &
        (lines["Source Substation"].isin(sub_names)) &
        (lines["Destination Substation"].isin(sub_names))
    ]
    return filtered_substations, filtered_lines

# Build the app and its layout

app = Dash(__name__)

app.layout = html.Div([
    html.H1("National Electricity Grid Network Analysis"),
    html.P("GridCare-Lite with ClinicCare-Lite — Course Final Project"),

    html.Div([
        html.Div([
            html.Label("Region"),
            dcc.Dropdown(region_list, region_list, id="region-filter", multi=True),
        ], style={"width": "32%", "display": "inline-block"}),

        html.Div([
            html.Label("Voltage (kV)"),
            dcc.Dropdown(voltage_list, voltage_list, id="voltage-filter", multi=True),
        ], style={"width": "32%", "display": "inline-block", "marginLeft": "2%"}),

        html.Div([
            html.Label("Utility"),
            dcc.Dropdown(utility_list, utility_list, id="utility-filter", multi=True),
        ], style={"width": "32%", "display": "inline-block", "marginLeft": "2%"}),
    ], style={"marginBottom": "20px"}),

    dcc.Tabs([
        dcc.Tab(label="Overview", children=[
            html.Div(id="kpi-cards", style={"display": "flex", "gap": "30px", "marginTop": "20px"}),
            dcc.Graph(id="region-bar"),
            dcc.Graph(id="voltage-bar"),
        ]),

        dcc.Tab(label="Network", children=[
            html.Div(id="network-stats", style={"marginTop": "20px"}),
            dcc.Graph(id="network-graph"),
            html.H4("Most Connected Substations"),
            dash_table.DataTable(id="network-table", page_size=10),
        ]),

        dcc.Tab(label="Geography", children=[
            html.Label("Color the map by:"),
            dcc.RadioItems(
                options=["Region", "Voltage (kV)"], value="Region",
                id="geo-color-choice", inline=True
            ),
            dcc.Graph(id="geo-map"),
        ]),

        dcc.Tab(label="Reliability", children=[
            dcc.Graph(id="status-bar"),
            dcc.Graph(id="capacity-bar"),
            dcc.Graph(id="age-hist"),
        ]),

        dcc.Tab(label="Search", children=[
            html.H4("Find a Substation"),
            dcc.Input(id="search-box", type="text", placeholder="Type a substation name..."),
            dash_table.DataTable(id="search-table", page_size=10),

            html.H4("Compare Two Utilities", style={"marginTop": "30px"}),
            html.Div([
                html.Div([
                    dcc.Dropdown(utility_list, utility_list[0], id="utility-1"),
                    html.Div(id="utility-1-output"),
                ], style={"width": "48%", "display": "inline-block"}),

                html.Div([
                    dcc.Dropdown(utility_list, utility_list[1] if len(utility_list) > 1 else utility_list[0], id="utility-2"),
                    html.Div(id="utility-2-output"),
                ], style={"width": "48%", "display": "inline-block", "marginLeft": "4%"}),
            ]),
        ]),
    ]),
])

#  OVERVIEW callback

@app.callback(
    Output("kpi-cards", "children"),
    Output("region-bar", "figure"),
    Output("voltage-bar", "figure"),
    Input("region-filter", "value"),
    Input("voltage-filter", "value"),
    Input("utility-filter", "value"),
)
def update_overview(regions, voltages, utils):
    filtered_substations, filtered_lines = filter_data(regions, voltages, utils)

    total_capacity = round(filtered_substations["Capacity (MVA)"].sum(), 1)

    def card(title, value):
        return html.Div([
            html.H3(str(value)),
            html.P(title),
        ], style={"border": "1px solid #ddd", "padding": "10px 20px", "borderRadius": "8px"})

    cards = [
        card("Substations", len(filtered_substations)),
        card("Lines", len(filtered_lines)),
        card("Total Capacity (MVA)", total_capacity),
        card("Utilities Selected", len(utils)),
    ]

    region_counts = filtered_substations["Region"].value_counts().reset_index()
    region_counts.columns = ["Region", "Count"]
    fig1 = px.bar(region_counts, x="Region", y="Count", color="Region", title="Substations by Region")

    voltage_counts = filtered_substations["Voltage (kV)"].value_counts().reset_index()
    voltage_counts.columns = ["Voltage (kV)", "Count"]
    fig2 = px.bar(voltage_counts, x="Voltage (kV)", y="Count", title="Substations by Voltage Level")

    return cards, fig1, fig2

#  NETWORK callback

@app.callback(
    Output("network-stats", "children"),
    Output("network-graph", "figure"),
    Output("network-table", "data"),
    Input("region-filter", "value"),
    Input("voltage-filter", "value"),
    Input("utility-filter", "value"),
)
def update_network(regions, voltages, utils):
    filtered_substations, filtered_lines = filter_data(regions, voltages, utils)

    G = nx.Graph()
    for index, row in filtered_substations.iterrows():
        G.add_node(row["Name"])
    for index, row in filtered_lines.iterrows():
        G.add_edge(row["Source Substation"], row["Destination Substation"])

    stats_text = f"Nodes (substations): {G.number_of_nodes()}  |  Edges (lines): {G.number_of_edges()}"

    if G.number_of_nodes() == 0:
        return stats_text, go.Figure(), []

    pos = nx.spring_layout(G, seed=42)

    edge_x, edge_y = [], []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1, color="gray"))

    node_x, node_y, node_labels = [], [], []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_labels.append(node)

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers", text=node_labels,
        hoverinfo="text", marker=dict(size=10, color="steelblue")
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(title="Substation Network Graph", showlegend=False, height=600)

    degree_rows = [[node, G.degree(node)] for node in G.nodes()]
    degree_df = pd.DataFrame(degree_rows, columns=["Substation", "Connections"])
    degree_df = degree_df.sort_values("Connections", ascending=False).head(10)

    return stats_text, fig, degree_df.to_dict("records")

# TAB GEOGRAPHY callback
@app.callback(
    Output("geo-map", "figure"),
    Input("region-filter", "value"),
    Input("voltage-filter", "value"),
    Input("utility-filter", "value"),
    Input("geo-color-choice", "value"),
)
def update_geo(regions, voltages, utils, color_choice):
    filtered_substations, filtered_lines = filter_data(regions, voltages, utils)

    fig = px.scatter_geo(
        filtered_substations, lat="Latitude", lon="Longitude",
        color=color_choice, size="Capacity (MVA)", hover_name="Name",
        hover_data=["Region", "Voltage (kV)", "Capacity (MVA)"],
        scope="africa", title="Substation Locations"
    )
    fig.update_layout(height=600)
    return fig

# RELIABILITY callback

@app.callback(
    Output("status-bar", "figure"),
    Output("capacity-bar", "figure"),
    Output("age-hist", "figure"),
    Input("region-filter", "value"),
    Input("voltage-filter", "value"),
    Input("utility-filter", "value"),
)
def update_reliability(regions, voltages, utils):
    filtered_substations, filtered_lines = filter_data(regions, voltages, utils)

    lines_with_region = filtered_lines.merge(
        filtered_substations[["Name", "Region"]],
        left_on="Source Substation", right_on="Name", how="left"
    )
    status_counts = lines_with_region.groupby(["Region", "Status"]).size().reset_index(name="Count")
    fig1 = px.bar(status_counts, x="Region", y="Count", color="Status", barmode="group",
                  title="Line Status by Region")

    avg_capacity = filtered_lines.groupby("Alias")["Capacity (MVA)"].mean().reset_index()
    fig2 = px.bar(avg_capacity, x="Alias", y="Capacity (MVA)", title="Average Line Capacity by Utility")

    filtered_substations = filtered_substations.copy()
    filtered_substations["Age"] = 2026 - filtered_substations["Commissioning Year"]
    fig3 = px.histogram(filtered_substations, x="Age", nbins=20, title="Substation Age Distribution")

    return fig1, fig2, fig3

#  SEARCH callback (substation finder)

@app.callback(
    Output("search-table", "data"),
    Input("search-box", "value"),
    Input("region-filter", "value"),
    Input("voltage-filter", "value"),
    Input("utility-filter", "value"),
)
def update_search(search_text, regions, voltages, utils):
    filtered_substations, filtered_lines = filter_data(regions, voltages, utils)

    if search_text:
        filtered_substations = filtered_substations[
            filtered_substations["Name"].str.contains(search_text, case=False)
        ]
    return filtered_substations.to_dict("records")

# SEARCH callback (utility comparison)

@app.callback(
    Output("utility-1-output", "children"),
    Output("utility-2-output", "children"),
    Input("utility-1", "value"),
    Input("utility-2", "value"),
)
def update_compare(utility1, utility2):
    def summary(utility_name):
        rows = lines[lines["Alias"] == utility_name]
        if len(rows) == 0:
            return html.P("No lines found for this utility.")
        return html.Div([
            html.P(f"Number of Lines: {len(rows)}"),
            html.P(f"Average Capacity (MVA): {round(rows['Capacity (MVA)'].mean(), 1)}"),
        ])
    return summary(utility1), summary(utility2)

#  Run the server

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8050, debug=False)

Writing app.py


In [ ]:

!python app.py &>/content/logs.txt &
import time
time.sleep(4)
!curl ipv4.icanhazip.com
!npx localtunnel --port 8050

136.110.60.136
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://wise-friends-sin.loca.lt
